# SAD Linac → BT → DR: RF-Track pipeline learning note

This notebook is a reproducible learning and integration check for the simulation-only ATF pipeline.  It tracks a 6D bunch from the post-accelerator Linac entrance through the SAD-daihon-derived Linac+BT lattice, applies an explicit BT→DR handoff, then tracks the DR turn-by-turn.  It makes no control-system connection.

The first Linac coordinate is **post-initial acceleration**: 82.36546122 MeV/c (the older simplified Linac interface uses 80 MeV/c).  With 76.14 MV cavity voltage, the SAD-derived reference particle exits Linac+BT at about 1300 MeV/c.  The current 1.3-GeV pipeline recalculates its IPZT optics with RF-Track at the configured energy and cavity voltage; it does not reuse the historical 1.542-GeV endpoint Twiss as though it were unchanged.  Its 1.3-GeV BT model preserves SAD normalised magnet strengths by ideal local field scaling (53 quadrupoles and 33 bends are retuned); this is not a record of measured operating currents.

## What is, and is not, established

Established: software-level 6D handoff, reference acceleration, finite-bunch tracking, and short DR turn tracking all execute in one path.  For the `sad_optics_matched` baseline, SAD `IPP1L` design Twiss is propagated through a centred RF-Track map at the configured Linac energy/RF voltage before matching to `RING0`; this removes the previous 1.542-GeV-to-1.3-GeV optics inconsistency.

Not established: physical injection transmission or final stored-beam transmission.  The SAD BT `CELLST` does include the historical septa and nominal `BK1R` kicker through `IPZT`; however, the handoff from that endpoint to periodic DR `RING0` is a zero-phase design-optics construction rather than a surveyed/calibrated map, and the model has no surveyed aperture/loss table.  Therefore `ring_survival` below must never be reported as measured or final transmission.

`dr_rf_mode='equilibrium'` is also available.  It finds the DR model's 6D synchronous orbit and places the relative Linac bunch time at that RF phase.  It intentionally retains the Linac–BT exit momentum, exposing rather than concealing injection-energy mismatch.  It is a deterministic model-phase alignment, not an ATF time-of-flight or RF calibration.

In [1]:
from pathlib import Path
import sys

# nbconvert starts a kernel beside this notebook, not necessarily at the repo root.
project_root = next(
    parent for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / 'Interfaces').is_dir()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from Interfaces.ATF2.ATF2_LinacBTDR_RFTrack import (
    ATF2LinacBTDRRFTrack, EntranceBunchTwiss,
)

# SAD IPP1L Twiss is propagated through the *configured* RF-Track Linac+BT
# before matching to RING0.  It is still not a surveyed injection map.
# 75.859978... MV is the offline model energy match, not an ATF setpoint.
machine = ATF2LinacBTDRRFTrack(
    cavity_voltage_mv=75.85997836493, dr_rf_mode='equilibrium',
    handoff_mode='sad_optics_matched',
)
machine.input_momentum_mev_c, machine.reference_exit, machine.dr_closed_orbit


RF-Track, version 2.6.3

Copyright (C) 2016-2026 CERN, Geneva, Switzerland. All rights reserved.

Author and contact:
 Andrea Latina <andrea.latina@cern.ch>
 BE-ABP Group
 CERN
 CH-1211 GENEVA 23
 SWITZERLAND

This software is distributed under a CERN proprietary software
license in the hope that it will be useful, but WITHOUT ANY WARRANTY;
not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

See the COPYRIGHT and LICENSE files at the top-level directory of
the RF-Track download area: https://gitlab.cern.ch/rf-track

RF-Track was compiled with GSL-2.5 and fftw-3.3.5-sse2-avx



[RF-Track] Could not check for updates.


(82.36546122,
 array([-3.91212255e-01, -4.79857117e-02,  9.86147504e-09,  2.25149797e-10]),
 array([-3.99900535e-01, -3.41103736e-01, -9.26130304e-19, -1.99158582e-18]))

In [2]:
# Response-matrix baseline for a corrector optimiser or RL comparison.
# Entries are derivatives of [x, xp, y, yp] at DR injection with respect
# to RF-Track-native, uncalibrated corrector strengths.
response, correctors = machine.linac_bt_dr_orbit_response_matrix(
    ('ZH1L', 'ZV1L', 'ZV2L'), step=1e-4
)
print(correctors)
print(response)

('ZH1L', 'ZV1L', 'ZV2L')
[[-1.57493175e+00  1.26554156e-06  1.55056690e-06]
 [-2.05901583e+00  1.07223175e-06  1.32527100e-06]
 [ 3.27674194e-12 -1.88149542e+00 -2.36158614e+00]
 [-3.03754255e-09  1.80070694e+00  9.98482837e-01]]


### RL environment contract

`LinacBTDRInjectionEnv` is dependency-free and has Gymnasium-like `reset` and `step` methods.  Its observation is the injection orbit relative to the DR closed orbit; its action is a bounded increment for each corrector; its reward is the negative normalised orbit error plus a small action penalty.  In addition to action clipping, `max_abs_delta_strength` bounds the cumulative native-strength excursion from the reset state.  This is a numerical RF-Track safety envelope—not an ATF power-supply limit—because extreme native values can abort the tracking process.  Training should use the endpoint reward, then call `validate_multi_turn(turn_history_sample_every=N)` only for selected policies.  This keeps DR long-turn tracking out of every training step.

In [3]:
from Interfaces.ATF2.linac_bt_dr_rl_env import LinacBTDRInjectionEnv

env = LinacBTDRInjectionEnv(max_steps=8)
observation, info = env.reset()
print('initial observation [mm, mrad, mm, mrad] =', observation)
print('actuators =', tuple(info['correctors_rftrack_strength']))
# An external PPO/SAC implementation can now choose an action in [-1, 1]^4.
# Do not use an all-zero action as a correction policy; this only exposes the API.

initial observation [mm, mrad, mm, mrad] = [ 5.49003115e-04 -4.20710336e-05  1.49934836e-04  5.52722961e-04]
actuators = ('ZH1L', 'ZH2L', 'ZV1L', 'ZV2L')


In [4]:
# Optional 6D DR RF/radiation model.  This takes a few seconds to find
# the model synchronous orbit.  It does not supply an injection calibration.
rf_machine = ATF2LinacBTDRRFTrack(dr_rf_mode='equilibrium')
rf_reference_exit = rf_machine.linac_bt_lattice.track(rf_machine.make_reference_bunch())
rf_reference_p = rf_reference_exit.get_phase_space()[0, 5]
print('DR synchronous [x, xp, y, yp, ct, p] =', rf_machine.dr_synchronous_orbit)
print('Linac–BT reference p minus synchronous p [MeV/c] =',
      rf_reference_p - rf_machine.dr_synchronous_orbit[5])

DR synchronous [x, xp, y, yp, ct, p] = [-3.99900535e-01 -3.41103736e-01 -9.26130304e-19 -1.99158582e-18
  1.15509775e+02  1.29552418e+03]
Linac–BT reference p minus synchronous p [MeV/c] = 4.478122166469575


The coordinates are `[x [mm], x' [mrad], y [mm], y' [mrad], t [mm/c], p [MeV/c]]`.  The factory below requires input Twiss explicitly.  `emitt_*` is normalised mm mrad, while `sigma_p_mev_c` is an absolute momentum spread.  Do not put `SAD_MARK_EMITX` etc. here without resolving their convention from a machine source or a separate validation.

The example uses `handoff_mode='sad_optics_matched'`: a zero-phase symplectic map that matches the RF-Track-calculated IPZT optics (seeded with SAD `IPP1L` Twiss at the configured Linac energy/RF voltage) and RFTrack `RING0$START` Twiss, and transforms first-order dispersion.  It is a useful design baseline for momentum-spread studies, but it has no surveyed coordinate transform, arbitrary phase is chosen as zero, and it contains no pulsed-kicker calibration.

In [5]:
# Demonstration only: this is not an ATF measured entrance distribution.
# Replace all values by a named operating point or a fitted measurement.
demo_twiss = EntranceBunchTwiss(
    emitt_x_norm_mm_mrad=1.0,
    emitt_y_norm_mm_mrad=1.0,
    beta_x_m=1.93, beta_y_m=1.93,
    sigma_t_mm_c=0.0, sigma_p_mev_c=0.0,
)
bunch = machine.make_entrance_bunch(demo_twiss, particles=16, charge_e=1.0e9)
result = machine.track(bunch, dr_turns=5, record_turn_history=True)
result.as_dict()

{'input': {'particles': 16,
  'charge_e': 1000000000.0,
  'survival_fraction_from_input': 1.0,
  'mean_x_mm': -0.013535575985727048,
  'mean_xp_mrad': 0.014535564061790773,
  'mean_y_mm': 0.02474033664922691,
  'mean_yp_mrad': 0.005119782616450731,
  'mean_p_mev_c': 82.36546122000001,
  'rms_x_mm': 0.10805725118277074,
  'rms_y_mm': 0.11010999539277481,
  'rms_p_mev_c': 1.4210854715202004e-14},
 'linac_bt_exit': {'particles': 16,
  'charge_e': 1000000000.0,
  'survival_fraction_from_input': 1.0,
  'mean_x_mm': 0.4729986945191348,
  'mean_xp_mrad': 0.08829226144941232,
  'mean_y_mm': -0.006196382869147553,
  'mean_yp_mrad': -2.575984461463383e-05,
  'mean_p_mev_c': 1295.5242510331184,
  'rms_x_mm': 4.677237744387188,
  'rms_y_mm': 2.0862413167813476,
  'rms_p_mev_c': 6.369858115312165e-05},
 'dr_injection': {'particles': 16,
  'charge_e': 1000000000.0,
  'survival_fraction_from_input': 1.0,
  'mean_x_mm': -0.39240486852841894,
  'mean_xp_mrad': -0.33255015008968625,
  'mean_y_mm': -7.41

## Reading the result

Compare `linac_bt_exit`, `dr_injection`, and each entry in `dr_turn_history`.  With the SAD IPP1L optics seed and the configured 1.3-GeV/RF map, the illustrative small-emittance study bunch has `mismatch_to_design` near one and survives the short-turn check; the standalone benchmark below reproduces this for 10 turns.  That validates internal optics consistency, not ATF injection efficiency.

The standalone command `python Interfaces/ATF2/diagnose_atf2_dr_energy_capture.py --turns 100` separates one important cause.  In the present model, the 76.14-MV Linac setting exits at 1300.0023 MeV/c while the RF synchronous momentum is 1295.5242 MeV/c; even the reference particle is lost at turn 57.  A secant match gives 75.859978 MV, an exit mismatch below 1e-7 MeV/c, and 100-turn reference survival.  This is a model diagnostic, **not an operational RF recommendation**, until RF phase, BT time of flight, injection energy, and pulsed kicker calibration are supplied.

For the response-matrix baseline, run `python -m Interfaces.ATF2.correct_linac_bt_dr_injection_orbit --turns 100`.  It adds synthetic `ZH5L/ZV5L` errors, uses four independent correctors and an SVD response-matrix solve to restore the DR injection orbit, then checks 100-turn survival.  It is an offline, uncalibrated reference implementation—not an emulation of real ATF errors.  The directly comparable first model-free baseline is `python -m Interfaces.ATF2.benchmark_linac_bt_dr_response_matrix_vs_rl --population 8 --generations 4 --validation-turns 10`.  Under the same synthetic error, four actuators, 8-step horizon, `1e-4` native-strength action scale, and `1e-3` cumulative numerical safety bound, it found an SVD response-matrix residual of `3.33e-8` normalised orbit norm in 13 endpoint interactions, versus `5.29e-3` for a small CEM stationary-policy search in 297 interactions; both survived ten model turns.  This is the expected result for a noiseless local-linear model: it establishes a baseline that RL must beat on a pre-registered nonlinear/noisy held-out test, not evidence that RL is superior.

For a fast scan, use `dr_turns=0` and compare exit/injection centroid, dispersion-subtracted projected Twiss, `mismatch_to_design`, and declared aperture margins after a real injection map is loaded.  The mismatch is 1 for an exactly matched uncoupled projected ellipse and increases above 1 for a mismatch.  Keep short multi-turn tracking as a rejection/validation stage.  `python -m Interfaces.ATF2.benchmark_linac_bt_dr_capture_proxy --particles 16 --short-turns 10` records both stages with an explicit SAD IPP1L optics seed and model-energy-matched RF.  Adding `--historical-kix-screen` applies only the historically reported 5-mm half-aperture at `KIX.1/.2`; its output explicitly says that arc, wiggler-mask, and south-straight constraints remain unmapped.  The companion `python -m Interfaces.ATF2.sweep_atf2_dr_injection_capture_proxy` gives a concrete proxy check: synthetic horizontal handoff offsets of 0.75, 1, and 2 mm first lost the reference particle on turns 3, 2, and 1 respectively, whereas 0.5 mm survived ten turns.  The 1-mm loss was unchanged with `--aperture-screen none`, so this is a prompt dynamic-capture threshold of the current model, **not** an attribution to KIX aperture.  This makes endpoint orbit useful for prompt-loss rejection, but not a final-transmission substitute.  Use a longitudinally synchronised ring plus a complete aperture map and a validated injection map before estimating final capture or damping-time survival.

On this environment, the 16-particle configured benchmark took about 0.37 s for endpoint-only tracking and 13.7 s for a fresh 10-turn RF/radiation check (the latter includes repetitive Linac+BT tracking), while construction including the RF-Track linearisation was about 6.5 s.  A four-particle 100-turn check took 26.3 s (about 0.262 s/turn) and survived with the explicitly partial KIX aperture screen.  A reference-particle 1000-turn direct check also completed in 14.9 s with survival at every 100-turn sample; it establishes long-run software continuity only, not finite-bunch transmission.  The nominal RF/radiation one-turn map has a roughly 72,000-turn 1/e damping time, so direct damping-time tracking would be hours even for that tiny bunch.  `--turn-history-sample-every N` bounds the retained long-turn history; its minimum survival is sampled rather than exact when N is above one.  Thus direct damping-time tracking is inappropriate for every optimiser proposal: use endpoint matching to filter candidates, short multi-turn tracking to reject prompt loss, and sparse long-horizon validation only after the injection map and apertures are calibrated.  A one-turn radiation-envelope/Lyapunov calculation can predict the stored-beam covariance efficiently, but cannot replace aperture-dependent capture/transmission tracking.

In [6]:
# Fast endpoint diagnostic versus a short multi-turn check.
endpoint = machine.track(bunch, dr_turns=0)
short = machine.track(bunch, dr_turns=5, record_turn_history=True)
print('endpoint survival:', endpoint.dr_injection.survival_fraction_from_input)
print('5-turn ring survival:', short.dr_after_turns.survival_fraction_from_input)
print('per-turn:', [s.survival_fraction_from_input for s in short.dr_turn_history])

endpoint survival: 1.0
5-turn ring survival: 1.0
per-turn: [1.0, 1.0, 1.0, 1.0, 1.0]


## Next calibration inputs

1. Before changing the model, run `python -m Interfaces.ATF2.validate_linac_bt_dr_end_to_end --long-turns 100 --long-sample-every 20`.  It is the software-consistency gate for energy matching, deterministic finite-bunch generation, optics handoff, 10-turn finite-bunch survival, and sparse 100-turn reference survival; it is not a physical transmission validation.
2. Supply a reconstructed post-accelerator Linac 6D bunch at the pipeline boundary, including RF phase/time and energy correlation.  The strict offline input path is now `python -m Interfaces.ATF2.validate_linac_bt_dr_direct_6d_input`; its schema requires `location: IPP1L`, `coordinate_order: x_mm,xp_mrad,y_mm,yp_mrad,t_mm_c,p_mev_c`, coordinates in `[mm, mrad, mm, mrad, mm/c, MeV/c]`, total `charge_e`, and provenance.  Use `python -m Interfaces.ATF2.benchmark_linac_bt_dr_capture_proxy --entrance-bunch-json measured-ippl-6d.json --short-turns 10` to compare the exact same distribution at endpoint and ring stages.  The bundled JSON is synthetic wiring data only; passing schema checks does not calibrate the diagnostic reconstruction.
3. The 80-MeV RF phase study is `python -m Interfaces.ATF2.scan_linac_klystron_phase_injection --coordinate-passes 1 --turns 10`.  It assumes the unverified pairing `L1→CA1L/CA2L` through `L8→CA15L/CA16L`, holds the nominal magnets and handoff fixed, and scans offsets relative to SAD's −1.67°-from-crest phase.  In the present model its zero-offset condition with 76.007891 MV per powered structure has a 1.35e-5 MeV/c DR-energy error and 10-turn survival; this is an offline model condition, not a klystron voltage/phase setpoint.  `--synthetic-starting-phase-error` demonstrates recovery from explicitly labelled artificial phase errors.
4. Replace the design `sad_optics_matched` handoff with a surveyed or fitted IPZT-to-RING0 map, including the pulsed kicker settings and dispersion.  `python -m Interfaces.ATF2.validate_linac_bt_dr_handoff_fit` validates the offline least-squares fitter: it requires at least five independent BT-endpoint/DR-injection dither pairs and can retain the separately calibrated baseline dispersion.  For real data, pass a JSON object containing `source_coordinates_mm_mrad`, `observed_target_coordinates_mm_mrad`, and optionally `baseline_dispersion_handoff` to `python -m Interfaces.ATF2.linac_bt_dr_handoff_fit data.json`; use its `handoff` result with `benchmark_linac_bt_dr_capture_proxy --handoff-json fit-result.json`.
5. Load measured/design apertures and loss locations.
6. Calibrate BT time-of-flight, DR RF phase, and injection energy for injected longitudinal motion.
7. Load an aperture table, for example `{'linac_bt': {'IPZT': [x_mm, y_mm]}, 'dr': {'QF2R.1': [x_mm, y_mm, 'rectangular']}}`, through `--aperture-json`.  These must be measured/design half-apertures, not guessed values.  The optional historical `KIX.1/.2` 5-mm screen is only a partial design cross-check, not a substitute; its circular cross-section is an explicit RF-Track study assumption because the historical source does not establish the chamber shape.
8. Calibrate against BPM, profile monitor, and injected/stored charge data; then compare endpoint metrics against multi-turn/final survival.  Do not use unbounded RF-Track-native corrector settings during exploratory scans: extreme values can abort the RF-Track process rather than reporting a clean loss.